## GTSRB dataset ke approximate numbers:

Train folder me ≈ 39,209 images

Test folder me ≈ 12,630 images

Total ≈ 51,839 images

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


In [2]:
from zipfile import ZipFile
with ZipFile("traffic_sign.zip", 'r') as zip_ref:
    zip_ref.extractall("gtsrb")


In [3]:
import numpy as np
from PIL import Image
import os

# Empty lists for images and labels
data = []
labels = []

# Loop through each class folder (0 to 42)
for class_id in range(43):
    path = f"gtsrb/Train/{class_id}"   # folder path
    images = os.listdir(path)

    for img_name in images:
        img_path = os.path.join(path, img_name)
        img = Image.open(img_path)         # open image
        img = img.resize((32,32))          # resize to 32x32
        img = np.array(img)                # convert to numpy array
        data.append(img)
        labels.append(class_id)

# Convert lists to numpy arrays
data = np.array(data)
labels = np.array(labels)

print("Images loaded:", data.shape)
print("Labels loaded:", labels.shape)


Images loaded: (39209, 32, 32, 3)
Labels loaded: (39209,)


In [4]:
print("First image shape:", data[0].shape)
print("First image label:", labels[0])


First image shape: (32, 32, 3)
First image label: 0


In [5]:
from tensorflow.keras.utils import to_categorical

# 1️⃣ Normalize images → pixels 0-1 me
data = data / 255.0

# 2️⃣ One-hot encode labels → 43 classes ke liye
labels = to_categorical(labels, 43)

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)


Data shape: (39209, 32, 32, 3)
Labels shape: (39209, 43)


In [6]:
model = Sequential([
Conv2D(32, (3,3),activation = 'relu', input_shape=(32,32,3)),
MaxPooling2D((2,2)),

Conv2D(64, (3,3), activation = 'relu'),
MaxPooling2D((2,2)),

Flatten(),

Dense(128, activation = 'relu'),
Dropout(0.5),

Dense(43, activation='softmax')
])

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 319,979 (1.22 MB)

 Trainable params: 319,979 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from sklearn.model_selection import train_test_split

# 20% data validation ke liye
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

# Train the model
history = model.fit(X_train, y_train,
                    batch_size=64,
                    epochs=10,
                    validation_data=(X_val, y_val))


Epoch 1/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.4504 - loss: 1.9723 - val_accuracy: 0.8470 - val_loss: 0.6361
Epoch 2/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7812 - loss: 0.6991 - val_accuracy: 0.9396 - val_loss: 0.2435
Epoch 3/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.8652 - loss: 0.4317 - val_accuracy: 0.9682 - val_loss: 0.1593
Epoch 4/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.8976 - loss: 0.3267 - val_accuracy: 0.9810 - val_loss: 0.0969
Epoch 5/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9155 - loss: 0.2635 - val_accuracy: 0.9843 - val_loss: 0.0778
Epoch 6/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9325 - loss: 0.2130 - val_accuracy: 0.9848 - val_loss: 0.0731
Epoch 7/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9375 - loss: 0.1934 - val_accuracy: 0.9861 - val_loss: 0.0592
Epoch 8/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9448 - loss: 0.1671 - val_accu

In [10]:
model.save("traffic_sign_model.h5")
